## install packages

In [1]:
%cd drive
%cd MyDrive
%cd intern

/content/drive
/content/drive/MyDrive
/content/drive/MyDrive/intern


In [2]:
# !sudo apt update
# !sudo apt install -y docker.io
# !sudo systemctl start docker
# !sudo systemctl enable docker


In [3]:
!pip install av
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.7/39.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 615.6 kB/s eta 0:00:00


In [4]:
# !docker --version
# !docker compose version
!pip install pymilvus
!pip install -qU langchain-milvus pymilvus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.3/248.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.5 MB/s eta 0:00:00


In [5]:
!pip install -qU langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00


In [6]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 30.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [7]:
from pymilvus import connections, DataType
from google.colab import userdata
# Connect using a MilvusClient object
from pymilvus import MilvusClient,Collection
from transformers import AutoProcessor, AutoModel,AutoTokenizer
# from langchain_milvus import Milvus
# from langchain_community.vectorstores import Milvus
from langchain_milvus.retrievers import MilvusCollectionHybridSearchRetriever
import json
import os
import torch
import numpy as np

## database creation

In [8]:
VIDEO_DIR = "./try_video/sample_0.mp4"
META_DIR = "./metadata"

In [9]:
# Retrieve your secrets
CLUSTER_ENDPOINT = userdata.get('CLUSTER_ENDPOINT')
TOKEN = userdata.get('TOKEN')
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

In [10]:
# Set your token

# Initialize a MilvusClient instance
# Replace uri and token with your own
client = MilvusClient(
    uri=CLUSTER_ENDPOINT, # Cluster endpoint obtained from the console
    token=TOKEN # API key or a colon-separated cluster username and password
)

In [11]:
def extract_video_metadata(json_filepath):
    """
    Extracts specific metadata (id, title, description, text_to_speech, characterList)
    from a video's JSON metadata file.

    Args:
        json_filepath (str): The path to the JSON metadata file.

    Returns:
        dict: A dictionary containing the extracted metadata, or None if an error occurs.
    """
    try:
        with open(json_filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # Extracting 'id' from 'original_json_filename' by removing the '.json' extension
        video_id = data.get('original_json_filename', 'N/A').replace('.json', '')

        # Extracting 'title' from 'content_metadata.title'
        title = data.get('content_metadata', {}).get('title', 'N/A')

        # Extracting 'description' from 'content_metadata.description'
        # Note: There's also youtube_description, you might choose one or combine them
        description = data.get('content_metadata', {}).get('description', 'N/A')

        # Extracting 'text_to_speech'
        text_to_speech = data.get('text_to_speech', 'N/A')

        # Extracting 'characterList'
        character_list = data.get('content_metadata', {}).get('characterList', [])

        extracted_data = {
            'id': video_id,
            'title': title,
            'description': description,
            'text_to_speech': text_to_speech,
            'characterList': character_list
        }
        return extracted_data

    except FileNotFoundError:
        print(f"Error: File not found at {json_filepath}")
        return None
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON format in {json_filepath}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None



In [12]:
json_filename = "metadata/sample_0.json"

# Path to your JSON file
# For your provided data, it would be 'sample_0.json' if saved locally.
# We'll use the dummy file for this example.
json_file = "metadata/sample_0.json"
extracted_info = extract_video_metadata(json_file)
if extracted_info:
    print("--- Extracted Video Metadata ---")
    print(f"ID: {extracted_info['id']}")
    print(f"Title: {extracted_info['title']}")
    print(f"Description: {extracted_info['description']}")
    print(f"Text to Speech (first 50 chars): {extracted_info['text_to_speech'][:50]}...")
    print("Character List:")
    for char in extracted_info['characterList']:
        print(f"  - Name: {char.get('name', 'N/A')}, Description: {char.get('description', 'N/A')}")
else:
    print("Failed to extract metadata.")


--- Extracted Video Metadata ---
ID: G_VTkkb34gw
Title: Life at the Academy
Description: A video analysis of the daily life and experiences of police recruits at the Queensland Police Service Academy.
Text to Speech (first 50 chars): [Music] [Applause] [Music] 154 yeah [Applause] Lif...
Character List:
  - Name: Police Recruit #1, Description: White female, light brown hair tied in a bun, pale complexion, wearing a blue QPS training shirt.
  - Name: Police Recruit #2, Description: White male with short brown hair, wearing a blue QPS training shirt.
  - Name: Joe Jaramazovic, Description: Superintendent, White male with short brown hair and greying temples, wearing a QPS uniform and tie.
  - Name: Jane Fitzgerald, Description: Senior Sergeant, White female with short blonde hair and glasses, wearing a QPS uniform.
  - Name: Unknown Officer 1, Description: White male, short brown hair, wearing a QPS uniform and tie.


In [13]:
# 1. Define the Schema
# Using a fixed ID for simplicity, but you can use auto_id=True for auto-generated IDs
# For characterList, we'll store it as a JSON string within a VARCHAR field for now.
# If your Milvus version supports DataType.JSON (newer versions), you can use that.
schema = MilvusClient.create_schema(
    auto_id=False, # Set to True if you want Milvus to generate IDs
    enable_dynamic_field=False # Setting to False for explicit schema fields
)


In [44]:
schema.add_field(field_name="video_id", datatype=DataType.VARCHAR, is_primary=True, max_length=256)
# Add a vector field. You'd typically generate this from your video/text data using an embedding model.
vector_dimension = 512 # Example dimension, replace with your actual embedding dimension
schema.add_field(field_name="video_embedding", datatype=DataType.FLOAT_VECTOR, dim=vector_dimension)
schema.add_field(field_name="text_embedding", datatype=DataType.FLOAT_VECTOR, dim=vector_dimension)
schema.add_field(field_name="video_filename", datatype=DataType.VARCHAR, max_length=512)
schema.add_field(field_name="description", datatype=DataType.VARCHAR, max_length=4096)

{'auto_id': False, 'description': '', 'fields': [{'name': 'video_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 256}, 'is_primary': True, 'auto_id': False}, {'name': 'video_embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 512}}, {'name': 'text_embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 512}}, {'name': 'video_filename', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 512}}, {'name': 'description', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 4096}}, {'name': 'video_id', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 256}, 'is_primary': True, 'auto_id': False}, {'name': 'video_embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 512}}, {'name': 'text_embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 512}}, {'name': 'v

In [15]:
# 3. Create the Collection
collection_name = "video_metadata_collection"
print(f"Checking if collection '{collection_name}' exists...")
if client.has_collection(collection_name=collection_name):
    print(f"Collection '{collection_name}' already exists. Dropping and recreating...")
    client.drop_collection(collection_name=collection_name)
    print("Collection dropped.")

print(f"Creating collection '{collection_name}'...")
client.create_collection(
    collection_name=collection_name,
    schema=schema,
    # index_params=index_params, # Remove index_params from create_collection
    # shards_num=2 # Example: You can set the number of shards
)
print(f"Collection '{collection_name}' created successfully.")


Checking if collection 'video_metadata_collection' exists...
Collection 'video_metadata_collection' already exists. Dropping and recreating...
Collection dropped.
Creating collection 'video_metadata_collection'...
Collection 'video_metadata_collection' created successfully.


In [16]:
# 4. Set up index
# 4.1. Set up the index parameters
index_params = client.prepare_index_params()

# 4.2. Add an index on the vector field.
index_params.add_index(
    field_name="video_embedding",
    metric_type="COSINE",
    index_type="AUTOINDEX",
    index_name="vector_index"
)


# 4.3. Add a second index on the 'text_embedding' field.
index_params.add_index(
    field_name="text_embedding",
    metric_type="COSINE",
    index_type="AUTOINDEX",
    index_name="text_vector_index" # Note: Added a new index_name
)
# 4.4. Create an index file
client.create_index(
    collection_name=collection_name ,
    index_params=index_params
)
# 5. Describe index
res = client.list_indexes(
    collection_name=collection_name
)
print(res)

['vector_index', 'text_vector_index']


In [17]:
def load_result(video_output_dir):
    # 加载元数据
    with open(os.path.join(video_output_dir, "metadata.json"), "r", encoding="utf-8") as f:
        metadata = json.load(f)

    # 加载嵌入向量
    video_emb = np.load(os.path.join(video_output_dir, metadata["embeddings"]["video_embeddings_path"]))
    text_emb = np.load(os.path.join(video_output_dir, metadata["embeddings"]["text_embeddings_path"]))

    return {
        "metadata": metadata,
        "video_embeddings": video_emb,
        "text_embeddings": text_emb
    }

# 使用
result = load_result("test-embedding/sample_0")
print(result["metadata"])
print("视频时长:", result["metadata"]["video_info"]["duration_seconds"], "秒")
print("视频嵌入形状:", result["video_embeddings"].shape)

{'video_path': '/root/autodl-tmp/X-CLIP/data/yidong/videos_compressed/sample_0.mp4', 'video_filename': 'sample_0.mp4', 'num_frames': 8, 'timestamps': [0.0, 38.33, 76.67, 115.0, 153.33, 191.67, 230.0, 268.33], 'frame_texts_info': [{'timestamp': 0.0, 'text': 'A video analysis of the daily life and experiences of police recruits at the Queensland Police Service Academy.', 'confidence': 1.0}, {'timestamp': 38.33, 'text': 'A video analysis of the daily life and experiences of police recruits at the Queensland Police Service Academy.', 'confidence': 1.0}, {'timestamp': 76.67, 'text': 'A video analysis of the daily life and experiences of police recruits at the Queensland Police Service Academy.', 'confidence': 1.0}, {'timestamp': 115.0, 'text': 'A video analysis of the daily life and experiences of police recruits at the Queensland Police Service Academy.', 'confidence': 1.0}, {'timestamp': 153.33, 'text': 'A video analysis of the daily life and experiences of police recruits at the Queensla

In [18]:
# 4. Prepare Data for Insertion
# Milvus expects a list of dictionaries, where each dictionary is an entity.
# For the `video_embedding` field, you'll need to generate an actual vector.
# For this example, we'll use a dummy vector.
# Convert characterList to a JSON string for storage in VARCHAR
# character_list_json_str = json.dumps(extracted_info['characterList'])

# Convert the torch tensor to a list of floats
# video_embedding_list = embed.squeeze().tolist()

i = 0
data_to_insert = [
    {
        "video_id": extracted_info['id'],
        # "title": extracted_info['title'],
        # "description": extracted_info['description'],
        # "text_to_speech": extracted_info['text_to_speech'],
        # "character_list_json": character_list_json_str,
        "video_embedding": np.squeeze(result["video_embeddings"]).tolist(),# Use the converted list of floats
        "text_embedding": np.squeeze(result["text_embeddings"]).tolist(),
        "description":extracted_info['description'],
        "video_filename": f"sample_{i}.mp4"
    }
]

In [19]:
# 5. Insert Data
print(f"Inserting data into '{collection_name}'...")
insert_result = client.insert(
    collection_name=collection_name,
    data=data_to_insert
)
print("Data insertion initiated.")
print(f"Insert count: {insert_result['insert_count']}")

# Important: Flush data to ensure it's queryable immediately
# In a production environment, you might let Milvus auto-flush or flush periodically.
# Flushing too often can lead to fragmented segments.
print("Flushing data...")
client.flush(collection_name=collection_name)
print("Data flushed.")

Inserting data into 'video_metadata_collection'...
Data insertion initiated.
Insert count: 1
Flushing data...
Data flushed.


## id retrival

In [20]:
# Retrieve video names by ID
# inserted_ids = results[0][0]['video_id']
inserted_ids = extracted_info['id']
print(f"Retrieved IDs: {inserted_ids}")

try:
    # Load the collection into memory for searching/querying
    client.load_collection(collection_name=collection_name)

    retrieved_entities = client.get(
        collection_name=collection_name,
        ids=inserted_ids,
        output_fields=["video_filename"]
        # output_fields=["video_id",  "video_embedding", "text_embedding","video_filename"] # Specify all fields you want to see
    )
    print(f'Retrieved Entities: {retrieved_entities}')
    print("Retrieved Entities:")
    for entity in retrieved_entities:
        print(entity)

finally:
    # Always release the collection when done to free up memory
    print(f"Releasing collection '{collection_name}' from memory.")
    client.release_collection(collection_name=collection_name)

Retrieved IDs: G_VTkkb34gw
Retrieved Entities: data: ["{'video_filename': 'sample_0.mp4', 'video_id': 'G_VTkkb34gw'}"], extra_info: {'cost': 6}
Retrieved Entities:
{'video_filename': 'sample_0.mp4', 'video_id': 'G_VTkkb34gw'}
Releasing collection 'video_metadata_collection' from memory.


## text retrival

In [21]:
model = AutoModel.from_pretrained("microsoft/xclip-base-patch32")
video_embd_func = model.get_video_features
tokenizer = AutoTokenizer.from_pretrained("microsoft/xclip-base-patch32")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/786M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

In [22]:
client.load_collection(collection_name=collection_name)
# The search parameters, which should match the index you've created on the collection
search_params = {
    "metric_type": "COSINE", # or "L2" or "IP"
    "params": {"nprobe": 10}, # Adjust nprobe based on your index type and needs
}
# Example query vector (replace with your actual embedding)
query_vector = np.random.rand(512).astype(np.float32).tolist()
# Perform the search on the video_embedding field
results = client.search(
    collection_name=collection_name,
    data=[query_vector],
    anns_field="video_embedding",
    limit=10,
    search_params=search_params,
    output_fields=["video_id", "video_filename"]
)

In [23]:
results

data: [[{'video_id': 'G_VTkkb34gw', 'distance': -0.004761697258800268, 'entity': {'video_filename': 'sample_0.mp4', 'video_id': 'G_VTkkb34gw'}}]],{'cost': 6}

In [26]:
from langchain_core.embeddings import Embeddings
from transformers import AutoTokenizer
from typing import List

# Load the tokenizer and model outside the class, so they are initialized only once
tokenizer = AutoTokenizer.from_pretrained("microsoft/xclip-base-patch32")
model = AutoModel.from_pretrained("microsoft/xclip-base-patch32")

class XClipEmbeddings(Embeddings):
    """
    Custom Embeddings class to wrap the Hugging Face X-CLIP model for LangChain.
    """
    def __init__(self, tokenizer, model):
        self.tokenizer = tokenizer
        self.model = model

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        """
        Embeds a list of documents.
        """
        # Tokenize the text inputs
        text_inputs = self.tokenizer(texts, padding=True, return_tensors="pt")
        # Get the text features from the model
        text_features = self.model.get_text_features(**text_inputs)
        # Return as a list of lists
        return text_features.tolist()

    def embed_query(self, text: str) -> List[float]:
        """
        Embeds a single text query.
        """
        # Tokenize the single text input
        text_inputs = self.tokenizer(text, padding=True, return_tensors="pt")
        # Get the text features from the model
        text_features = self.model.get_text_features(**text_inputs)
        # Return the first (and only) embedding as a single list
        return text_features.tolist()[0]

# Initialize the custom class with the pre-loaded tokenizer and model
xclip_embeddings = XClipEmbeddings(tokenizer, model)

## text RAG

In [27]:
# Example: Embed a list of documents
documents = ["a photo of a cat", "a man walking in the park"]
doc_embeddings = xclip_embeddings.embed_documents(documents)

# Example: Embed a single query
query = "a picture of a dog"
query_embedding = xclip_embeddings.embed_query(query)

print("Document embeddings shape:", len(doc_embeddings), len(doc_embeddings[0]))
print("Query embedding shape:", len(query_embedding))

Document embeddings shape: 2 512
Query embedding shape: 512


In [28]:
ZILLIZ_CLOUD_URI = 'https://in03-b510f824c907c11.serverless.aws-eu-central-1.cloud.zilliz.com'
ZILLIZ_CLOUD_USERNAME = 'db_b510f824c907c11'
ZILLIZ_CLOUD_PASSWORD = 'Vk8>1JOsDpAcD-q|'

In [29]:
from langchain_community.vectorstores import Milvus
# Connect to the existing collection
vector_db = Milvus(
    embedding_function =xclip_embeddings,
    connection_args={
        "uri": ZILLIZ_CLOUD_URI,
        "user": ZILLIZ_CLOUD_USERNAME,
        "password": ZILLIZ_CLOUD_PASSWORD,
        "secure": True,
    },
    collection_name=collection_name,
    vector_field="video_embedding",
    text_field="description"
)

/tmp/ipython-input-1197694674.py:3: LangChainDeprecationWarning: The class `Milvus` was deprecated in LangChain 0.2.0 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-milvus package and should be used instead. To use it run `pip install -U :class:`~langchain-milvus` and import as `from :class:`~langchain_milvus import MilvusVectorStore``.
  vector_db = Milvus(


In [36]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Convert the Milvus vector store into a retriever
# This makes it a standard LangChain component that can be used in a chain
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

# Define a prompt template for your RAG system
template = """
You are an AI assistant. Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Context: {context}
Question: {question}
"""
prompt = PromptTemplate.from_template(template)

# Define your LLM
llm = ChatOpenAI(model="gpt-5-nano",api_key=OPENAI_API_KEY)

# Build the RAG chain
# The retriever will fetch the most relevant context, and the LLM will generate a response
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [37]:
# Assuming 'retriever' has been defined as in the original code
query = "What is Retrieval-Augmented Generation?"
retrieved_docs = retriever.invoke(query)

# Print the number of documents retrieved
print(f"Retrieved {len(retrieved_docs)} documents.")

# Iterate and print the content and metadata of each document
for i, doc in enumerate(retrieved_docs):
    print(f"\n--- Document {i+1} ---")
    print(f"Content: {doc.page_content[:200]}...")  # Print the first 200 characters
    print(f"Metadata: {doc.metadata}")

Retrieved 1 documents.

--- Document 1 ---
Content: A video analysis of the daily life and experiences of police recruits at the Queensland Police Service Academy....
Metadata: {'video_id': 'G_VTkkb34gw', 'text_embedding': [-0.045583367347717285, -0.032575029879808426, -0.023312892764806747, 0.06409826129674911, 0.0022140934597700834, -0.01804332807660103, -0.004655770491808653, -0.0215937327593565, 0.06258057802915573, -0.019803361967206, -0.0095022888854146, -0.036452699452638626, 0.05649494752287865, -0.017649605870246887, 0.021360522136092186, 0.030600139871239662, -0.05884687229990959, 0.024557286873459816, -0.00948928203433752, -0.0011739827459678054, 0.038296233862638474, -0.0154662961140275, 0.02944573573768139, -0.044224124401807785, 0.028989039361476898, 0.008849985897541046, -0.017664076760411263, 0.03972402215003967, 0.018066495656967163, 0.03112686425447464, 0.004718433599919081, -0.019134024158120155, -0.012133457697927952, -0.0022951511200517416, 0.019622523337602615, 0.

In [38]:
user_query = "What do these images show?"
# Invoke the chain with a user query
response = rag_chain.invoke(user_query)

print(response)

They show police recruits at the Queensland Police Service Academy, illustrating their daily life and training experiences.


## video RAG

In [33]:
import base64
from io import BytesIO
from PIL import Image

def pil_to_base64(image: Image) -> str:
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

In [ ]:
import av
import os
def extract_frames_by_count(video_path, num_frames=8):
    container = av.open(video_path)
    stream = container.streams.video[0]
    total_frames = stream.frames

    if total_frames < num_frames:
        print("Video has fewer than 8 frames.")
        return []

    interval = max(1, total_frames // num_frames)
    frames = []

    for i, frame in enumerate(container.decode(stream)):
        if len(frames) == num_frames:
            break
        if i % interval == 0:
            frames.append(frame.to_image())

    container.close()
    return frames
path = 'videos'
video = results[0][0]['video_filename']
video_path = os.path.join(path,video)
# Usage example:
extracted_images = extract_frames_by_count(video_path)

In [34]:
# Assume 'images' is your list of PIL.Image.Image objects
base64_images = [pil_to_base64(img) for img in extracted_images]

In [35]:
from langchain_core.messages import HumanMessage

# Your text query for the LLM
text_query = "What do these images show?"

# Combine text and images into a single content list
content_list = [{"type": "text", "text": text_query}]
for img_base64 in base64_images:
    content_list.append(
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_base64}"}}
    )

# Create the HumanMessage object
message = HumanMessage(content=content_list)

In [ ]:
video_rag_chain = llm | StrOutputParser()

In [ ]:
response = video_rag_chain.invoke([message])

In [ ]:
print(response)

They’re scenes from a police training academy (Queensland Police Service). Specifically:
- A large group of recruits lined up outside a campus building.
- An indoor training session or briefing with instructors.
- An outdoor parade/ceremony with officers in formation.
- A female officer speaking or being interviewed near a sign for the academy.
